In [1]:
import sqlite3 as sql
from typing import Dict, Iterator
import pandas as pd
import numpy as np
import os
from pathlib import Path
import re
import time

### カテゴリ変数を俯瞰する

In [ ]:
src_conn = sql.connect('test_db/homogenous_data.db')

In [35]:
src_cursor = src_conn.cursor()
src_cursor.execute("PRAGMA table_info('homogenous_table')")
original_cols = [row[1] for row in src_cursor.fetchall()]
print(original_cols)
extract_cols = [col for col in original_cols if any(keyword in str(col) for keyword in ("Name", "Nm"))]
print(extract_cols)
remain_cols = [col for col in original_cols if col not in extract_cols]
print(remain_cols)

['Date', 'Code', 'CoName', 'CoNameEn', 'S17', 'S17Nm', 'S33', 'S33Nm', 'ScaleCat', 'Mkt', 'MktNm', 'Mrgn', 'MrgnNm', 'O', 'H', 'L', 'C', 'UL', 'LL', 'Vo', 'Va', 'AdjFactor', 'AdjO', 'AdjH', 'AdjL', 'AdjC', 'AdjVo', 'MO', 'MH', 'ML', 'MC', 'MUL', 'MLL', 'MVo', 'MVa', 'MAdjO', 'MAdjH', 'MAdjL', 'MAdjC', 'MAdjVo', 'AO', 'AH', 'AL', 'AC', 'AUL', 'ALL', 'AVo', 'AVa', 'AAdjO', 'AAdjH', 'AAdjL', 'AAdjC', 'AAdjVo']
['CoName', 'CoNameEn', 'S17Nm', 'S33Nm', 'MktNm', 'MrgnNm']
['Date', 'Code', 'S17', 'S33', 'ScaleCat', 'Mkt', 'Mrgn', 'O', 'H', 'L', 'C', 'UL', 'LL', 'Vo', 'Va', 'AdjFactor', 'AdjO', 'AdjH', 'AdjL', 'AdjC', 'AdjVo', 'MO', 'MH', 'ML', 'MC', 'MUL', 'MLL', 'MVo', 'MVa', 'MAdjO', 'MAdjH', 'MAdjL', 'MAdjC', 'MAdjVo', 'AO', 'AH', 'AL', 'AC', 'AUL', 'ALL', 'AVo', 'AVa', 'AAdjO', 'AAdjH', 'AAdjL', 'AAdjC', 'AAdjVo']


In [18]:
def get_unique_values(column_name: str) -> list:
    src_cursor.execute(
        f"""
        SELECT DISTINCT {column_name}
        FROM homogenous_table
        """
    )
    unique_values = [row[0] for row in src_cursor.fetchall()]
    unique_values.sort()
    return unique_values

In [36]:
S17_unique_values = get_unique_values("S17Nm")
S33_unique_values = get_unique_values("S33Nm")
ScaleCat_unique_values = get_unique_values("ScaleCat")
Mkt_Unique_values = get_unique_values("MktNm")
Mrgn_Unique_values = get_unique_values("MrgnNm")

In [37]:
print(f"Length: {len(S17_unique_values)}")
print(S17_unique_values)

Length: 18
['その他', 'エネルギー資源', '不動産', '医薬品', '商社・卸売', '小売', '建設・資材', '情報通信・サービスその他', '機械', '素材・化学', '自動車・輸送機', '運輸・物流', '金融（除く銀行）', '鉄鋼・非鉄', '銀行', '電機・精密', '電気・ガス', '食品']


In [38]:
print(f"Length: {len(S33_unique_values)}")
print(S33_unique_values)

Length: 34
['その他', 'その他製品', 'その他金融業', 'ガラス･土石製品', 'ゴム製品', 'サービス業', 'パルプ・紙', '不動産業', '保険業', '倉庫･運輸関連業', '化学', '医薬品', '卸売業', '小売業', '建設業', '情報･通信業', '機械', '水産・農林業', '海運業', '石油･石炭製品', '空運業', '精密機器', '繊維製品', '証券･商品先物取引業', '輸送用機器', '金属製品', '鉄鋼', '鉱業', '銀行業', '陸運業', '電気機器', '電気･ガス業', '非鉄金属', '食料品']


In [39]:
print(f"Length: {len(ScaleCat_unique_values)}")
print(ScaleCat_unique_values)

Length: 6
['-', 'TOPIX Core30', 'TOPIX Large70', 'TOPIX Mid400', 'TOPIX Small 1', 'TOPIX Small 2']


In [40]:
print(f"Length: {len(Mrgn_Unique_values)}")
print(Mkt_Unique_values)

Length: 3
['JASDAQ グロース', 'JASDAQ スタンダード', 'TOKYO PRO MARKET', 'その他', 'グロース', 'スタンダード', 'プライム', 'マザーズ', '東証一部', '東証二部']


In [41]:
print(len(Mrgn_Unique_values))
print(Mrgn_Unique_values)

3
['その他', '信用', '貸借']


In [43]:
src_conn.close()

### DropOrigin_OneHotENcoded_homogenous : <strong>Analysis</strong>

In [3]:
con = sql.connect('test_db/homogenous_data.db')
cur = con.cursor()

In [4]:
def get_column_values(cursor: sql.Cursor) -> list:
    cursor.execute(
        f"""
        SELECT * FROM DropOrigin_OneHotEncoded_homogenous LIMIT 0
        """
    )
    column_names = [desc[0] for desc in cursor.description]
    return column_names

In [6]:
print(get_column_values(cur))

['Date', 'Code', 'O', 'H', 'L', 'C', 'UL', 'LL', 'Vo', 'Va', 'AdjFactor', 'AdjO', 'AdjH', 'AdjL', 'AdjC', 'AdjVo', 'MO', 'MH', 'ML', 'MC', 'MUL', 'MLL', 'MVo', 'MVa', 'MAdjO', 'MAdjH', 'MAdjL', 'MAdjC', 'MAdjVo', 'AO', 'AH', 'AL', 'AC', 'AUL', 'ALL', 'AVo', 'AVa', 'AAdjO', 'AAdjH', 'AAdjL', 'AAdjC', 'AAdjVo', 'S17_1', 'S17_2', 'S17_3', 'S17_4', 'S17_5', 'S17_6', 'S17_7', 'S17_8', 'S17_9', 'S17_10', 'S17_11', 'S17_12', 'S17_13', 'S17_14', 'S17_15', 'S17_16', 'S17_17', 'S17_18', 'S33_1', 'S33_2', 'S33_3', 'S33_4', 'S33_5', 'S33_6', 'S33_7', 'S33_8', 'S33_9', 'S33_10', 'S33_11', 'S33_12', 'S33_13', 'S33_14', 'S33_15', 'S33_16', 'S33_17', 'S33_18', 'S33_19', 'S33_20', 'S33_21', 'S33_22', 'S33_23', 'S33_24', 'S33_25', 'S33_26', 'S33_27', 'S33_28', 'S33_29', 'S33_30', 'S33_31', 'S33_32', 'S33_33', 'S33_34', 'ScaleCat_1', 'ScaleCat_2', 'ScaleCat_3', 'ScaleCat_4', 'ScaleCat_5', 'ScaleCat_6', 'MktNm_1', 'MktNm_2', 'MktNm_3', 'MktNm_4', 'MktNm_5', 'MktNm_6', 'MktNm_7', 'MktNm_8', 'MktNm_9', 'Mkt

In [8]:
con.close()

### NullFilled_DropOrigin_OneHotEncoded_homogenous: <strong>Null Filling</strong>

In [4]:
con = sql.connect('test_db/homogenous.sqlite3')
cur = con.cursor()